In [ ]:
import keras
import datetime
import tensorflow                       as tf
from tensorflow.keras.callbacks         import TensorBoard
from tensorflow.keras.layers            import Input,Lambda,UpSampling2D,Conv2D,Dropout,MaxPooling2D,Conv2DTranspose,concatenate,BatchNormalization, Activation,ConvLSTM2D,TimeDistributed, GlobalAveragePooling2D, GlobalMaxPooling2D, Add, multiply,Reshape, LayerNormalization
from tensorflow.keras.models            import Model
from tensorflow.keras.optimizers        import Adam,RMSprop,SGD
from keras.utils                        import plot_model
from tensorflow.keras                   import layers, models
from tensorflow.keras.losses            import mae
from tensorflow.keras.callbacks         import LearningRateScheduler,Callback

from sklearn.metrics                    import mean_squared_error,r2_score,mean_absolute_error

import sys
import os
import numpy as np
import math
import time
import random, time
from pathlib                        import Path
from PIL                            import Image

import skimage                      as ski
from   skimage.filters              import threshold_otsu
from   skimage                      import io, color
from   skimage.color                import rgb2gray
from   skimage                      import filters
import cv2                          as cv
import matplotlib.pyplot            as plt 
import gc
import glob
from skimage                        import img_as_ubyte
from skimage                        import io
import shutil
import pandas as pd
tf.keras.backend.clear_session()

### 1. Dataset Creation and Loading

In [ ]:
#directory = '/home/guiomar/Desktop/CODES/predicting-flow-patterns'
directory = '/home/ppgi/Trabajo/predicting-flow-patterns'

d1=directory+'/G_Masked'
d2=directory+'/P_Masked'
d3=directory+'/V_Masked'
d4=directory+'/Vx_Masked'
d5=directory+'/Vy_Masked'

gtrain= d1+'/train'
gtest=d1+'/test'
gval=d1+'/valid'

ptrain=d2+'/train'
ptest=d2+'/test'
pval=d2+'/valid'

vtrain=d3+'/train'
vtest=d3+'/test'
vval=d3+'/valid'

vxtrain=d4+'/train'
vxtest=d4+'/test'
vxval=d4+'/valid'

vytrain=d5+'/train'
vytest=d5+'/test'
vyval=d5+'/valid'

nbatch = 10

In [ ]:
def dataset_array(g_path,p_path,v_path,vx_path,vy_path):
    """
    Loads and preprocesses a single set of data files.
    This function is designed to be used with tf.data.Dataset.map().
    """
    def load_array(geo,p,v,vx,vy):
        '''
        Inner function executed by tf.numpy_function.
        It loads numpy arrays from the provided file paths
        '''
        pre = np.load(p)   
        vel = np.load(v)
        velx = np.load(vx)
        vely = np.load(vy)
        g = np.load(geo)
        y = np.concatenate([pre, vel, velx, vely], axis=-1)
        return g,y
    
    x,y= tf.numpy_function(load_array,[g_path,p_path,v_path, vx_path,vy_path], [tf.float64,tf.float64])

    y.set_shape([y.shape[0],y.shape[1],y.shape[2]])
    x.set_shape([x.shape[0],x.shape[1],x.shape[2]])
  
    return x,y 


def create_dataset(p_path,v_path, vx_path,vy_path,g_path,batch_size = nbatch):
    p_files = sorted(glob.glob(os.path.join(p_path, "*.npy")))
    v_files = sorted(glob.glob(os.path.join(v_path, "*.npy")))
    vx_files = sorted(glob.glob(os.path.join(vx_path, "*.npy")))
    vy_files = sorted(glob.glob(os.path.join(vy_path, "*.npy")))
    g_files = sorted(glob.glob(os.path.join(g_path, "*.npy")))

    dataset = tf.data.Dataset.from_tensor_slices((p_files,v_files,vx_files,vy_files,g_files))
    dataset = dataset.map(dataset_array, num_parallel_calls=tf.data.AUTOTUNE)
    dataset = dataset.batch(batch_size)
    return dataset

train_ds =create_dataset(gtrain,ptrain,vtrain,vxtrain,vytrain)
test_ds  =create_dataset(gtest,ptest,vtest,vxtest,vytest)
valid_ds =create_dataset(gval,pval,vval,vxval,vyval)

### 2. Hyperparameters

In [ ]:
num_epochs    =50
patience      =10     # How long to wait after last time validation loss improved
LR            =0.001

# Model name
model_names=['ConvLSTM']
NamesKeras=['Arch_1.keras','Arch_2.keras','Arch_3.keras','Arch_4.keras','Arch_5.keras']
model_name    =model_names[0]
save_in      ='/home/guiomar/Desktop/CODES/predicting-flow-patterns'
#save_in       ='/home/ppgi/Trabajo/predicting-flow-patterns'

# image dimensions
img_width     =  256   # 739   G:737
img_height    =  64   # 185
channel       =  1

number_of_filters = [8,16,32,64,128,256,512]

type_padding = 'same'
f_activation = 'relu'
f_activation_last='relu'
DECAY_RATE=0.04

### 3. Exponential Decay Learning Rate

In [ ]:
def exponential_decay(epoch,lr_ini=LR,decay_rate=DECAY_RATE,epochs=num_epochs):
    if epoch < epochs*0.001:
        return lr_ini
    else:
        return  lr_ini * np.exp(-decay_rate*epoch)
    
epochs = np.arange(num_epochs)
learning_rates = [round(exponential_decay(epoch),5) for epoch in epochs]


plt.plot(epochs, learning_rates, label="Learning Rate")
plt.title("Exponential Decay of the Learning Rate")
plt.xlabel("Epoch")
plt.ylabel("Learning Rate")
plt.grid(False)
plt.show()  

### 4. Callbacks

In [ ]:
lr_scheduler = LearningRateScheduler(lambda epoch: exponential_decay(epoch), verbose=1)
optimizer = Adam(learning_rate=LR)
early_stopping = tf.keras.callbacks.EarlyStopping(monitor='val_loss',  min_delta=1e-2, patience=patience, restore_best_weights=True,verbose=1)
save = save_in + '/Test5_Results_weights/Test1_Best_weights.weights.h5'
checkpoint_weight=tf.keras.callbacks.ModelCheckpoint(save,save_weights_only=True)
checkpoint_keras = tf.keras.callbacks.ModelCheckpoint(NamesKeras[0],save_best_only=True)


In [ ]:

#Optional 
def MSEplus(u_true, u_pred):
    
  Eps=1e-4
  
  Ip_value = tf.reduce_sum(tf.cast(u_true != 0.0, tf.float64)) 
  u_true = tf.cast(u_true, dtype=tf.float64)
  u_pred = tf.cast(u_pred, dtype=tf.float64)  
  E_normL2=tf.sqrt(tf.reduce_sum(tf.square(u_true - u_pred)))
  E=tf.reduce_sum(tf.square(u_true - u_pred))
  u_pred_normL2=tf.sqrt(tf.reduce_sum(tf.square(u_pred)))
  mse_loss = tf.reduce_sum(E + E_normL2 / (u_pred_normL2 + Eps))
  mse_loss=mse_loss/Ip_value
 
  return mse_loss 

In [71]:


def make_model(output_channels=4):
    
    inputs1 = Input((img_height, img_width, channel))
    inputs = layers.Reshape((1,img_height, img_width, channel))(inputs1)
    # Encoder (contracting path)
    x1 = layers.ConvLSTM2D(filters=32, kernel_size=(3,3), padding='same',
                           return_sequences=True, activation='tanh')(inputs)
    x1 = layers.BatchNormalization()(x1)
    print(x1.shape)
    x2 = layers.ConvLSTM2D(filters=64, kernel_size=(3,3), padding='same',
                           return_sequences=True, activation='tanh')(x1)
    x2 = layers.BatchNormalization()(x2)
    print(x2.shape)
    x3 = layers.ConvLSTM2D(filters=128, kernel_size=(3,3), padding='same',
                           return_sequences=True, activation='tanh')(x2)
    x3 = layers.BatchNormalization()(x3)
    print(x3.shape)
    # Bottleneck
    bottleneck = layers.ConvLSTM2D(filters=256, kernel_size=(3,3), padding='same',
                                   return_sequences=True, activation='tanh')(x3)
    bottleneck = layers.BatchNormalization()(bottleneck)
    print(bottleneck.shape)
    print('**********')
    # Decoder (expanding path) + skip connections
    d1 = layers.ConvLSTM2D(filters=128, kernel_size=(3,3), padding='same',
                           return_sequences=True, activation='tanh')(bottleneck)
    print(d1.shape)
    d1 = layers.Concatenate()([d1, x3])
    print('concat',d1.shape)
    d1 = layers.BatchNormalization()(d1)

    d2 = layers.ConvLSTM2D(filters=64, kernel_size=(3,3), padding='same',
                           return_sequences=True, activation='tanh')(d1)
    print(d2.shape)
    d2 = layers.Concatenate()([d2, x2])
    print('concat',d2.shape)
    d2 = layers.BatchNormalization()(d2)
   

    d3 = layers.ConvLSTM2D(filters=32, kernel_size=(3,3), padding='same',
                           return_sequences=True, activation='tanh')(d2)
    print(d3.shape)
    d3 = layers.Concatenate()([d3, x1])
    print('concat',d3.shape)
    d3 = layers.BatchNormalization()(d3)

    # Output layer: campo de flujo (u, v)
    outputs = layers.Conv3D(filters=output_channels, kernel_size=(1,1,1), 
                            padding='same', activation='linear')(d3)
    print(outputs.shape)

    model = keras.Model(inputs=inputs, outputs=outputs)
    model.summary()

    model.compile(optimizer=optimizer, 
              loss = ['mean_squared_error'],                     # 'mean_squared_error'
              
              metrics= ['mae', "root_mean_squared_error"] )
    return model


### 5. Architecture

In [72]:
M=make_model()

(None, 1, 64, 256, 32)
(None, 1, 64, 256, 64)
(None, 1, 64, 256, 128)
(None, 1, 64, 256, 256)
**********
(None, 1, 64, 256, 128)
concat (None, 1, 64, 256, 256)
(None, 1, 64, 256, 64)
concat (None, 1, 64, 256, 128)
(None, 1, 64, 256, 32)
concat (None, 1, 64, 256, 64)
(None, 1, 64, 256, 4)


Model: "functional_10"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ keras_tensor_395CL… │ (None, 1, 64,     │          0 │ -                 │
│ (InputLayer)        │ 256, 1)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv_lstm2d_78      │ (None, 1, 64,     │     38,144 │ keras_tensor_395… │
│ (ConvLSTM2D)        │ 256, 32)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 1, 64,     │        128 │ conv_lstm2d_78[1… │
│ (BatchNormalizatio… │ 256, 32)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv_lstm2d_79      │ (None, 1, 64,     │    221,440 │ batch_normalizat… │
│ (ConvLSTM2D)        │ 256, 64)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 1, 64,     │        256 │ conv_lstm2d_79[1… │
│ (BatchNormalizatio… │ 256, 64)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv_lstm2d_80      │ (None, 1, 64,     │    885,248 │ batch_normalizat… │
│ (ConvLSTM2D)        │ 256, 128)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 1, 64,     │        512 │ conv_lstm2d_80[1… │
│ (BatchNormalizatio… │ 256, 128)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv_lstm2d_81      │ (None, 1, 64,     │  3,539,968 │ batch_normalizat… │
│ (ConvLSTM2D)        │ 256, 256)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 1, 64,     │      1,024 │ conv_lstm2d_81[1… │
│ (BatchNormalizatio… │ 256, 256)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv_lstm2d_82      │ (None, 1, 64,     │  1,769,984 │ batch_normalizat… │
│ (ConvLSTM2D)        │ 256, 128)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_33      │ (None, 1, 64,     │          0 │ conv_lstm2d_82[1… │
│ (Concatenate)       │ 256, 256)         │            │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 1, 64,     │      1,024 │ concatenate_33[1… │
│ (BatchNormalizatio… │ 256, 256)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv_lstm2d_83      │ (None, 1, 64,     │    737,536 │ batch_normalizat… │
│ (ConvLSTM2D)        │ 256, 64)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_34      │ (None, 1, 64,     │          0 │ conv_lstm2d_83[1… │
│ (Concatenate)       │ 256, 128)         │            │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 1, 64,     │        512 │ concatenate_34[1… │
│ (BatchNormalizatio… │ 256, 128)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv_lstm2d_84      │ (None, 1, 64,     │    184,448 │ batch_normalizat… │
│ (ConvLSTM2D)        │ 256, 32)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_35      │ (None, 1, 64,     │          0 │ conv_lstm2d_84[1

 Total params: 7,380,740 (28.16 MB)

 Trainable params: 7,378,884 (28.15 MB)

 Non-trainable params: 1,856 (7.25 KB)

### 6. Parameters Number

In [73]:
params_simple = M.count_params()
print('Parameters Number: ', params_simple)

Parameters Number:  7380740


### 7. Fitting

In [74]:
print("Starting trainig")
start_time_for_fit = time.time()
history = M.fit(train_ds,epochs=num_epochs,validation_data = valid_ds,callbacks=[lr_scheduler,early_stopping,checkpoint_weight,checkpoint_keras])
end_time_for_fit = time.time()
Training_time = end_time_for_fit - start_time_for_fit
print(f"TRAINING TIME IN  {Training_time:.4f} segundos.")
    

Starting trainig

Epoch 1: LearningRateScheduler setting learning rate to 0.001.
Epoch 1/50


NotImplementedError: numpy() is only available when eager execution is enabled.

### 8. Evaluation

In [ ]:
results = M.evaluate(test_ds) 
print("Evaluation results:")
for name, value in zip(M.metrics_names, results):
    print(f"{name}: {value:.4f}")

### 9. Prediction

In [ ]:
xtrue=[]
ytrue=[]
ypred = []
for x,y in test_ds:
    xtrue.append(x.numpy())
    ytrue.append(y.numpy())
 
start_time_pred=time.time()  
for i in range(len(ytrue)):    
    ypred.append(M.predict(xtrue[i]))
end_time_pred=time.time() 

inference_time = end_time_pred - start_time_pred


print(f"PREDICTING TIME IN  {inference_time:.4f} seconds.")

In [ ]:
# Metrics
y_preds = []
y_trues = []

for x_batch, y_batch in test_ds:
    y_pred = M.predict(x_batch)
    y_preds.append(y_pred)
    y_trues.append(y_batch)
    
y_preds = np.concatenate(y_preds, axis=0)  # (N, H, W, 4)
y_trues = np.concatenate(y_trues, axis=0)

# MSE, MAE, RMSE by channel
for i, var in enumerate(["p", "v", "vx", "vy"]):
    mse = mean_squared_error(y_trues[..., i].ravel(), y_preds[..., i].ravel())
    mae = mean_absolute_error(y_trues[..., i].ravel(), y_preds[..., i].ravel())
    rmse = np.sqrt(mse)
    print(f"{var}: MSE = {mse:.4f}, MAE = {mae:.4f}, RMSE ={rmse:.4f}")


y_true_flat = y_trues.flatten()
y_pred_flat = y_preds.flatten()

mse = mean_squared_error(y_true_flat, y_pred_flat)
mae = mean_absolute_error(y_true_flat, y_pred_flat)
rmse = np.sqrt(mse)
maepercent=mae*100
print(f"MSE global: {mse:.6f}")
print(f"MAE global: {mae:.6f}")
print(f"MAE % global: {maepercent:.6f}")
print(f"RMSE global: {rmse:.6f}")


### 10. Figures

In [ ]:
pd.DataFrame(history.history)[['loss', 'val_loss']].plot(figsize=(8, 5))
plt.yscale('log')       # Escala logarítmica para el eje Y
plt.grid(True, which='both')  # Mostrar grid tanto en escala mayor como menor
plt.title("Loss vs Epoch")
plt.xlabel("Epochs")
plt.ylabel("Loss (log)")
plt.tight_layout()
plt.show()

In [ ]:

pd.DataFrame(history.history)[['mae', 'val_mae']].plot(figsize=(8, 5))
plt.yscale('log')       # Escala logarítmica para el eje Y
plt.grid(True, which='both')  # Mostrar grid tanto en escala mayor como menor
plt.title("Mae vs Epochs")
plt.xlabel("Epochs")
plt.ylabel("Mae (log)")
plt.tight_layout()
plt.show()

In [ ]:

pd.DataFrame(history.history)[['root_mean_squared_error', 'val_root_mean_squared_error']].plot(figsize=(8, 5))
       # Escala logarítmica para el eje Y
plt.grid(True, which='both')  # Mostrar grid tanto en escala mayor como menor
plt.title("RMSE vs Epochs")
plt.xlabel("Epochs")
plt.ylabel(" RMSE")
plt.tight_layout()
plt.show()

### 11. Qualitative Visualization

In [ ]:
def plotting():
    n = random.randint(0, len(xtrue)-1)
    m = random.randint(0, nbatch-1)
    titles = [ 'P pred', 'P true','V pred', 'V true','Vx pred', 'Vx true','Vy pred', 'Vy true']
    fig, axes = plt.subplots(4, 3, figsize=(16, 10))
    
    error=[]
    for i in range(4):
            error.append(np.abs(ytrue[m][n][:,:,i]- ypred[m][n][:,:,i]))

            axes[i,0].imshow(ypred[m][n][:,:,i], cmap='gray')
            axes[i, 0].set_title(titles[2*i])
            axes[i, 0].axis('off')

            axes[i,1].imshow(ytrue[m][n][:,:,i], cmap='gray')
            axes[i, 1].set_title(titles[2*i+1])
            axes[i, 1].axis('off')

            e=axes[i,2].imshow(error[0], cmap='turbo')
            axes[i, 2].set_title('Absolute Error')
            axes[i, 2].axis('off')
            plt.colorbar(e, ax=axes[i,2], fraction=0.046)

    for ax in axes.flat:
        ax.axis('off')

In [ ]:
plotting()

In [ ]:
plotting()

In [ ]:
plotting()

In [ ]:
plotting()